#  Semantic Matching with Gemini

**Learning Objectives**
  1. Learn how use embeddings model by python client
  2. Learn how to calculate different simlarity metricks
  3. Learn how to identify matching products by using embeddings generated from descriptions
  1. Learn how to design a prompt for semantic matching
  2. Learn how to evaluate performance of a prompt for semantic matching
  1. Learn how to use Gemini with Google Gen AI SDK
  
Semantic matching is the problem of classifying a pair of entities $(x_1, x_2)$ as being a good match or not. So it is a classification setup that is a very flexible: Namely, it comprises general information retrieval (where the first entity can be a textual query and the second entity can be a paragraph for instance), entity resolutions, or database-record fuzzy-matching. In this notebook we will focus on matching textual descriptions of retail products. More specifically:
  
  
**Use case description:** An online retail company scours the web to compare prices of products in their inventory with those offered by their competitors. Their first priority is to implement a model that compares the information on two product webpages and outputs a classification indicating whether the different product descriptions on the webpages correspond to an identical product, which we will refer to as a 'match'. We use the [Amazon-Google Products dataset](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) in the [entity resolution benchmark](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) created by Leipzig University. 

**Model description:** This notebook illustrates how to use Gemini API in Agent Platform to match product descriptions. The [Amazon-GoogleProducts dataset](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) contains product information about the products scrapped on Amazon or Google websites. It includes the products title, description, price, and manufacturer, athough this information is worded differently on the two websites. In this notebook, we will focus solely on building a model using the product titles. The idea is straightforward: we will create a prompt that asks the Gemini API whether the product titles match or not. Although the information about the products is limited to these titles, we still achieve an accuracy close to 100% on the test set.

<!-- **Evaluation method:** In order to avoid overfitting the prompt design on our dataset, we first split the our dataset sample of paired descriptions into an evaluation set (evalset) containing 60 examples and a test set (testset) also consisting of 60 examples. The choice of 60 examples aligns with the current limit quota of 60 requests per minute for the Gemini API in Agent Platform. Subsequently, we devise the prompt using the evalset and report the model's accuracy on the testset. Both the evaluation and test sets are roughly balanced. -->

## Setup

In [ ]:
import random
import os
import numpy as np
import pandas as pd
from google import genai
import os
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig

pd.options.display.max_colwidth = 1000

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]
BUCKET = f"{PROJECT}-cord19-semantic-search"

# Do not change these
os.environ["PROJECT"] = PROJECT
os.environ["BUCKET"] = BUCKET
os.environ["REGION"] = REGION

### Define the embedding model:

In [ ]:
EMBEDDING_MODEL = "text-embedding-005"

## Exploring the dataset

### Initialize the client:
In our case we using Agent Platfrom: vertexai=True and provide your Google Cloud Project ID and Region

In [ ]:
client = genai.Client(
    vertexai=True,
    project=PROJECT,
    location="global"
)

#### Lets generate embedding from text

TODO: Explain task_type="SEMANTIC_SIMILARITY" 
# Generate the embedding
Define utility method to create embedding from text string:

In [ ]:
def embed_text(text_to_embed: str | list[str], output_dimensionality: int = 768) -> list[list[float]]:
    """
    Generates embeddings for the provided text.
    """
    try:
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text_to_embed,
            config=types.EmbedContentConfig(
                task_type="SEMANTIC_SIMILARITY",
                output_dimensionality=output_dimensionality 
            )
        )
        
        return [np.array(embed.values) for embed in response.embeddings]

    except Exception as e:
        print(f"Failed to generate embeddings: {e}")

### Example on how to use embed_text method:

In [ ]:
text_to_embed = "emc retrospect 7.5 disk to diskcromwindows"

test_embedding = embed_text(text_to_embed)
print(f"Generated a vector with {len(test_embedding[0])} dimensions.")
print(f"View the first 5 floats: {test_embedding[0][:5]}") # View the first 5 floats
input_embd_vector=test_embedding[0]

### Calculate the L2 norm (Euclidean magnitude)

In [ ]:
magnitude = np.linalg.norm(input_embd_vector)
print(f"vector magnitude = {magnitude}")

### L2 Normalization Note
Important Normalization Rule: The API automatically L2-normalizes the default 768-dimensional vector. However, if you use output_dimensionality to request a smaller size (like 128 or 768), the API returns raw, unnormalized numbers. You must manually normalize the truncated vector in your code before you use it to compute cosine similarity.

In [ ]:
embedding_mrl_128 = embed_text(text_to_embed, output_dimensionality=128)
print(f"Generated a vector with {len(embedding_mrl_128[0])} dimensions.")
magnitude = np.linalg.norm(embedding_mrl_128)
print(f"vector magnitude = {magnitude}")

### Just trancate it

In [ ]:
embedding_trunc_128 = input_embd_vector[:128]
print(f"Generated a vector with {len(embedding_trunc_128)} dimensions.")
magnitude = np.linalg.norm(embedding_trunc_128)
print(f"vector magnitude = {magnitude}")

### Manual L2 Normalization

In [ ]:
def l2_normalize(vec):
    """Applies L2 normalization to a vector."""
    return vec / np.linalg.norm(vec)

In [ ]:
embedding_l2_norm_mrl_128 = l2_normalize(embedding_trunc_128)
magnitude = np.linalg.norm(embedding_l2_norm_mrl_128)
print(f"vector magnitude = {magnitude}")

### Visualize embedding:

In [ ]:
import plotly.express as px
import numpy as np

# Plot the vector as a sequence of bars
fig = px.bar(
    x=range(len(input_embd_vector)),
    y=input_embd_vector,
    # Crucial change: pass the values to 'color' to map them to a continuous scale
    color=input_embd_vector, 
    # Use a diverging color scale (Red for positive, Blue for negative)
    color_continuous_scale="RdBu_r", 
    labels={"x": f"Vector Dimension (0 - {len(input_embd_vector)})", "y": "Activation Value", "color": "Value"},
    title="Vector Visualization (Color-Coded Activation)",
    template="plotly_white"
)

# Style the bars to be sharp and distinct
fig.update_traces(marker_line_width=0)
fig.update_layout(
    bargap=0.1,
    coloraxis_cmid=0 
)

fig.show()

### Load first dataset with product descriptions

We use a [dataset of product information](https://dbs.uni-leipzig.de/file/Amazon-GoogleProducts.zip) scraped from Google and Amazon websites. The dataset is part of a [benchmark for semantic matching and entity resolution](https://dbs.uni-leipzig.de/research/projects/object_matching/benchmark_datasets_for_entity_resolution) from Leipzig University. It contains 3 tables which are included in this repo:

```python
../data/Amazon.csv.gz 
../data/GoogleProducts.csv.gz
../Amzon_GoogleProducts_perfectMapping.csv.gz

```

The first table contains product information listed on Amazon, including the product title, description, and manufacturer:

In [ ]:
amazon = pd.read_csv("../data/Amazon.csv.gz")
amazon.columns = [
    "idAmazon",
    "amazon_title",
    "amazon_description",
    "amazon_manufacturer",
    "amazon_price",
]
amazon = amazon.dropna(subset=['amazon_description'])
amazon.head()

### Lets create embeddings for concatenated title and description with a space in between

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

# 1. Define your batch size
BATCH_SIZE = 20

# Concatenate Title and Description with a space in between
combined_text = amazon['amazon_title'].fillna('') + " " + amazon['amazon_description'].fillna('') #+ amazon['amazon_manufacturer'].fillna('')

# Convert to list and apply the truncation logic we used earlier
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]

all_embeddings_768 = []

print(f"Starting to embed {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. Loop through the list in chunks
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # Slice the list to get the current batch
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # Call embed_text function
    batch_embeddings_768 = embed_text(batch_texts)
    
    # Safety check: if the API call failed and returned an empty list, 
    # pad with None so our final list length matches the DataFrame length.
    if not batch_embeddings_768:
        print(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings_768 = [None] * len(batch_texts)
        
    all_embeddings_768.extend(batch_embeddings_768)
    
    # Pause briefly to avoid hitting Requests-Per-Minute (RPM) limits.
    # Adjust this value based on your specific API quota.
    time.sleep(1.0) 

# 4. Assign the resulting embeddings back to the DataFrame
if len(all_embeddings_768) == len(amazon):
    amazon['amazon_embedding_768'] = all_embeddings_768
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(batch_embeddings_768)} embeddings for {len(amazon)} rows.")

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

In [ ]:
import numpy as np
from numpy.linalg import norm

def calculate_cosine_similarity(embedding1, embedding2):
    """
    Calculates cosine similarity between two embedding vectors.
    Formula: (A · B) / (||A|| * ||B||)
    """ 
    # Calculate the dot product and the magnitudes
    dot_product = np.dot(embedding1, embedding2)
    magnitude = norm(embedding1) * norm(embedding2)
    
    # Avoid division by zero in case of an empty vector
    if magnitude == 0:
        return 0.0
        
    return dot_product / magnitude

### Example Usage
1. Generate an embedding for your search query (using our first function)
2. Apply the similarity function across the entire column
3. Sort the DataFrame to see the most relevant results at the top

In [ ]:
amazon['cosine_similarity'] = amazon['amazon_embedding_768'].apply(
    lambda row_embedding: calculate_cosine_similarity(input_embd_vector, row_embedding)
)

amazon = amazon.sort_values(by='cosine_similarity', ascending=False)
# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### Visualize Similarity distribution
Plot the histogram of similarity distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")

# Initialize the figure
plt.figure(figsize=(10, 6))

sns.histplot(
    amazon['cosine_similarity'],
    bins=100,
    kde=False,
    color='royalblue'
)

# Add titles and labels for readability
plt.title('Distribution of Cosine Similarity Scores', fontsize=14, pad=15)
plt.xlabel('Cosine Similarity (Higher = More Similar)', fontsize=12)
plt.ylabel('Number of Products', fontsize=12)

# Display the plot
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Create a sample of a 10 top most similar records
sampled_df = amazon.iloc[:20]

# Stack the individual embedding arrays into a single 2D matrix and extract input vector
embedding_matrix = np.vstack(sampled_df['amazon_embedding_768'].values) - input_embd_vector

# 5. Plot the heatmap
plt.figure(figsize=(16, 4)) # Wide and short layout matching your image

# Create the heatmap
sns.heatmap(
    embedding_matrix,
    cmap='RdBu_r',     # Red-Blue diverging colormap
    center=0,          # Forces 0 to be pure white
    xticklabels=False, # Removes x-axis labels for a cleaner look
    yticklabels=False, # Removes y-axis labels
    cbar=True          # Shows the color scale on the right
)

plt.title("Embedding Vectors Difference (Sorted by Similarity)", pad=15)
plt.xlabel("Embedding Dimensions")
plt.ylabel("Products (Most Similar -> Least Similar)")

plt.show()

### Remove cosine_similarity column

In [ ]:
amazon = amazon.drop(columns=["cosine_similarity"])

Calculate the L2 norm (Euclidean magnitude)
Lsts check is it L2 normalized

### 

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

amazon['amazon_embedding_128'] = [
    embed[:128] for embed in amazon['amazon_embedding_768'].tolist()
]

amazon['amazon_embedding_l2norm_128'] = [
    l2_normalize(embed[:128]) for embed in amazon['amazon_embedding_768'].tolist()
]

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

### Load second dayaset
The second table contains the same information but for product information scrapped from Google website:

In [ ]:
google = pd.read_csv("../data/GoogleProducts.csv.gz")
google.columns = [
    "idGoogleBase",
    "google_title",
    "google_description",
    "google_manufacturer",
    "google_price",
]
google = google.dropna(subset=['google_description'])
google.head()

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter

# 1. Define your batch size
BATCH_SIZE = 20

# 2. Extract the descriptions as a standard Python list
#descriptions = google['google_description'].tolist()
combined_text = google['google_title'].fillna('') + " " + google['google_description'].fillna('')

# 2. Convert to list and apply the truncation logic we used earlier
MAX_CHARS = 4000
descriptions = [str(text)[:MAX_CHARS] for text in combined_text.tolist()]
all_embeddings = []

print(f"Starting to generate embeddings for {len(descriptions)} documents in batches of {BATCH_SIZE}...")

# 3. Loop through the list in chunks
for i in tqdm(range(0, len(descriptions), BATCH_SIZE), desc="Embedding Batches"):
    # Slice the list to get the current batch
    batch_texts = descriptions[i : i + BATCH_SIZE]
    
    # Call your optimized embed_text function
    batch_embeddings = embed_text(batch_texts)
    
    # Safety check: if the API call failed and returned an empty list, 
    # pad with None so our final list length matches the DataFrame length.
    if not batch_embeddings:
        logging.warning(f"Batch {i} to {i + BATCH_SIZE} failed. Padding with None.")
        batch_embeddings = [None] * len(batch_texts)
        
    all_embeddings.extend(batch_embeddings)
    
    # Pause briefly to avoid hitting Requests-Per-Minute (RPM) limits.
    # Adjust this value based on your specific API quota.
    time.sleep(1.0) 

# 4. Assign the resulting embeddings back to the DataFrame
if len(all_embeddings) == len(google):
    google['google_embedding_768'] = all_embeddings
    print("Successfully added embeddings to the DataFrame!")
else:
    print(f"Length mismatch: {len(all_embeddings)} embeddings for {len(google)} rows.")



import pandas as pd

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(google.head())

In [ ]:
import time
import logging
from tqdm.auto import tqdm # Great for showing a progress bar in Jupyter
import pandas as pd

google['google_embedding_128'] = [
    embed[:128] for embed in google['google_embedding_768'].tolist()
]

google['google_embedding_l2norm_128'] = [
    l2_normalize(embed[:128]) for embed in google['google_embedding_768'].tolist()
]

# Temporarily restrict column width to 100 characters
with pd.option_context('display.max_colwidth', 100):
    display(amazon.head())

In [ ]:
# amazon = amazon.drop(columns=["cosine_similarity"])
# with pd.option_context('display.max_colwidth', 100):
#     display(amazon.head())

In [ ]:
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")
#matching.head()
matching = matching.merge(right=google, how="left", on="idGoogleBase")
matched_products = matching.merge(right=amazon, how="left", on="idAmazon")
matched_products = matched_products.dropna(subset=['amazon_description'])
matched_products = matched_products.dropna(subset=['google_description'])
# google_descriptions = list(matched_products["google_title"])
# amazon_descriptions = list(matched_products["amazon_title"])

# # Create the dataframe of matching descriptions
# matching_descriptions = pd.DataFrame(
#     {
#         "description_1": google_descriptions,
#         "description_2": amazon_descriptions,
#         "match": "yes",
#     }
with pd.option_context('display.max_colwidth', 100):
    display(matched_products.head())

### Lets use semantic search

The similarity_func now expects exactly two single values (e.g., two strings or two 1D arrays) to evaluate at a time. This is indeed much slower, but it is the perfect setup for micro-benchmarking or profiling how efficiently a specific distance or similarity algorithm performs per call.

In [ ]:
import time
import numpy as np
import pandas as pd

def evaluate_matching_ranked(df, col_amazon, col_google, similarity_func, is_distance=False):
    """
    Evaluates matches between Amazon and Google columns and ranks all combinations.
    
    Returns:
    - result_df: DataFrame with appended ranking lists (1 to N) and Top-K metrics.
    """
    result_df = df.copy()
    
    amazon_data = result_df[col_amazon].tolist()
    google_data = result_df[col_google].tolist()
    
    # Extract the ground truth IDs to map back to our predictions
    google_ids = result_df['idGoogleBase'].values 
    
    N = len(amazon_data)
    M = len(google_data)
    
    metric_type = "distance" if is_distance else "similarity"
    print(f"Calculating {metric_type} iteratively for {N} x {M} ({N * M:,}) combinations...")
    
    similarity_matrix = np.zeros((N, M))
    start_time = time.perf_counter()

    # --- Full Iteration Logic ---
    # We compare every Amazon item to every Google item.
    for i in range(N):
        for j in range(M): 
            similarity_matrix[i, j] = similarity_func(amazon_data[i], google_data[j])
            
    end_time = time.perf_counter()

    # --- Calculate Performance Metrics ---
    execution_time = end_time - start_time
    
    # --- Ranking Logic ---
    # np.argsort returns the indices that would sort the array. 
    if is_distance:
        # For distances, lower is better (ascending sort)
        ranked_indices = np.argsort(similarity_matrix, axis=1)
    else:
        # For similarities, higher is better (descending sort)
        # We use negative similarity_matrix to sort highest-to-lowest
        ranked_indices = np.argsort(-similarity_matrix, axis=1)

    # Pre-allocate lists for our new DataFrame columns
    all_ranked_ids = []
    all_ranked_scores = []
    correct_match_ranks = []

    for i in range(N):
        # Extract the sorted indices for this specific Amazon row
        row_sorted_indices = ranked_indices[i]
        
        # Reorder the IDs and Scores based on the sorted indices
        sorted_scores = similarity_matrix[i, row_sorted_indices]
        sorted_ids = google_ids[row_sorted_indices]
        
        all_ranked_ids.append(sorted_ids.tolist())
        all_ranked_scores.append(sorted_scores.tolist())
        
        # Calculate what rank the actual correct ID achieved (1-based indexing)
        true_id = result_df['idGoogleBase'].iloc[i]
        rank_pos = np.where(sorted_ids == true_id)[0]
        
        if len(rank_pos) > 0:
            correct_match_ranks.append(rank_pos[0] + 1) # +1 so rank starts at 1, not 0
        else:
            correct_match_ranks.append(None)
            
    # --- Assigning to DataFrame ---
    result_df['ranked_predicted_idGoogleBase'] = all_ranked_ids
    result_df['ranked_scores'] = all_ranked_scores
    result_df['correct_match_rank'] = correct_match_ranks
    
    # Preserve original logic: isolate the #1 best match
    result_df['predicted_idGoogleBase'] = [ids[0] for ids in all_ranked_ids]
    result_df['best_match_score'] = [scores[0] for scores in all_ranked_scores]
    result_df['is_correct_match'] = result_df['idGoogleBase'] == result_df['predicted_idGoogleBase']
    
    # Only calculate the diagonal pair score if the matrix is perfectly square
    if N == M:
        result_df['pair_score'] = similarity_matrix.diagonal()
    else:
        result_df['pair_score'] = None
    
    total_comparisons = N * M
    
    # Calculate Top-K Accuracies
    top_1_acc = (result_df['correct_match_rank'] == 1).mean() * 100
    top_3_acc = (result_df['correct_match_rank'] <= 3).mean() * 100
    top_5_acc = (result_df['correct_match_rank'] <= 5).mean() * 100
    top_10_acc = (result_df['correct_match_rank'] <= 10).mean() * 100
    top_20_acc = (result_df['correct_match_rank'] <= 20).mean() * 100
    
    print("-" * 40)
    print(f"Execution time: {execution_time:.4f} seconds")
    print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
    print(f"Average time per comparison: {(execution_time / total_comparisons) * 1e6:.2f} microseconds")
    print("-" * 40)
    print("Model Accuracy (Where did the correct match rank?):")
    print(f"Best Match Accuracy: {top_1_acc:.2f}%")
    print(f"Top-3 Accuracy: {top_3_acc:.2f}%")
    print(f"Top-5 Accuracy: {top_5_acc:.2f}%")
    print(f"Top-10 Accuracy: {top_10_acc:.2f}%")
    print(f"Top-20 Accuracy: {top_20_acc:.2f}%")
    print("-" * 40)
    
    return result_df

In [ ]:
# Function takes two individual 1D arrays
def cosine_sim_item(vec1, vec2):
    norm_product = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if norm_product == 0:
        return 0.0
    return np.dot(vec1, vec2) / norm_product

def dot_product(vec1, vec2):
    return np.dot(vec1, vec2)

import numpy as np

def euclidean_similarity_item(vec1, vec2):
    """
    Calculates the Euclidean similarity between two 1D vectors.
    Returns a score between 0.0 and 1.0 (where 1.0 is an exact match).
    """
    # Ensure the inputs are NumPy arrays (in case they are passed as standard lists)
    v1 = np.asarray(vec1)
    v2 = np.asarray(vec2)
    
    # 1. Calculate the Euclidean distance
    # np.linalg.norm calculates the square root of the sum of squared differences
    distance = np.linalg.norm(v1 - v2)


vector_results1 = evaluate_matching_ranked(
    df=matched_products, 
    col_amazon='amazon_embedding_768', 
    col_google='google_embedding_768', 
    similarity_func=cosine_sim_item
)

# vector_results1 = evaluate_matching_ranked(
#     df=matched_products, 
#     col_amazon='amazon_embedding_128', 
#     col_google='google_embedding_128', 
#     similarity_func=cosine_sim_item
# )
    
# vector_results2 = evaluate_matching_ranked(
#     df=matched_products, 
#     col_amazon='amazon_embedding_768', 
#     col_google='google_embedding_768', 
#     similarity_func=dot_product
# )

# vector_results2 = evaluate_matching_ranked(
#     df=matched_products, 
#     col_amazon='amazon_embedding_128', 
#     col_google='google_embedding_128', 
#     similarity_func=dot_product
# )

# vector_results2 = evaluate_matching_ranked(
#     df=matched_products, 
#     col_amazon='amazon_embedding_768', 
#     col_google='google_embedding_768',
#     similarity_func=euclidean_similarity_item,
#     is_distance=True
# )

# vector_results2 = evaluate_matching(
#     df=matched_products, 
#     col_amazon='amazon_embedding', 
#     col_google='google_embedding', 
#     similarity_func=euclidean_similarity_item,
#     is_distance=True
# )

In [ ]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Stack the embedding lists into 2D NumPy arrays (Matrices)
amazon_matrix = np.vstack(matched_products['amazon_embedding'].values)
google_matrix = np.vstack(matched_products['google_embedding'].values)

print(f"Calculating similarities for {amazon_matrix.shape[0]} x {google_matrix.shape[0]} matrix...")

# Start the high-resolution timer
start_time = time.perf_counter()

# 2. Calculate pairwise cosine similarity for all combinations at once
#similarity_matrix = cosine_similarity(amazon_matrix, google_matrix)

# Perform the dot product using matrix multiplication
# .T transposes the google matrix to shape (768, N_google)

similarity_matrix = amazon_matrix @ google_matrix.T

# Stop the timer
end_time = time.perf_counter()

# 3. Find the index and score of the highest similarity score for each Amazon row
best_match_indices = np.argmax(similarity_matrix, axis=1)
best_match_scores = np.max(similarity_matrix, axis=1)

# 4. Extract the predicted Google IDs using the found indices
# Since google_matrix was built from matched_products, the row indices align perfectly
predicted_google_ids = matched_products['idGoogleBase'].iloc[best_match_indices].values
predicted_google_titles = matched_products['google_title'].iloc[best_match_indices].values

# 5. Assign predictions to the DataFrame
matched_products['predicted_idGoogleBase'] = predicted_google_ids
matched_products['predicted_google_title'] = predicted_google_titles
matched_products['best_match_score'] = best_match_scores

dot_products = np.sum(amazon_matrix * google_matrix, axis=1)

# 3. Calculate the magnitudes (norms) of each vector
norm_amazon = np.linalg.norm(amazon_matrix, axis=1)
norm_google = np.linalg.norm(google_matrix, axis=1)

# 4. Calculate cosine similarity (safely handling division by zero)
denominators = norm_amazon * norm_google
pair_scores = np.divide(
    dot_products, 
    denominators, 
    out=np.zeros_like(dot_products), 
    where=denominators != 0
)

# 5. Assign the results to the new column
matched_products['pair_score'] = pair_scores

# 6. Validate the match (Does the predicted ID match the actual ground truth ID?)
matched_products['is_correct_match'] = matched_products['idGoogleBase'] == matched_products['predicted_idGoogleBase']

# Calculate and display performance metrics
execution_time = end_time - start_time
total_comparisons = amazon_matrix.shape[0] * google_matrix.shape[0]
accuracy = matched_products['is_correct_match'].mean() * 100



print("-" * 40)
print(f"Execution time: {execution_time:.4f} seconds")
print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
print(f"Model Accuracy: {accuracy:.2f}% ({matched_products['is_correct_match'].sum()} correct out of {len(matched_products)})")
print("-" * 40)

# View the results to see where the model succeeded and failed
columns_to_show = [
    'idAmazon', 
    'idGoogleBase', 
    'predicted_idGoogleBase', 
    'is_correct_match', 
    'best_match_score',
    'pair_score'
]

matched_products[columns_to_show].head(10)

In [ ]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Stack the embedding lists into 2D NumPy arrays (Matrices)
amazon_matrix = np.vstack(matched_products['amazon_embedding'].values)
google_matrix = np.vstack(matched_products['google_embedding'].values)

print(f"Calculating similarities for {amazon_matrix.shape[0]} x {google_matrix.shape[0]} matrix...")

# Start the high-resolution timer
start_time = time.perf_counter()

# 2. Calculate pairwise cosine similarity for all combinations at once
#similarity_matrix = cosine_similarity(amazon_matrix, google_matrix)

# Perform the dot product using matrix multiplication
# .T transposes the google matrix to shape (768, N_google)

similarity_matrix = amazon_matrix @ google_matrix.T

# Stop the timer
end_time = time.perf_counter()

# 3. Find the index and score of the highest similarity score for each Amazon row
best_match_indices = np.argmax(similarity_matrix, axis=1)
best_match_scores = np.max(similarity_matrix, axis=1)

# 4. Extract the predicted Google IDs using the found indices
# Since google_matrix was built from matched_products, the row indices align perfectly
predicted_google_ids = matched_products['idGoogleBase'].iloc[best_match_indices].values
predicted_google_titles = matched_products['google_title'].iloc[best_match_indices].values

# 5. Assign predictions to the DataFrame
matched_products['predicted_idGoogleBase'] = predicted_google_ids
matched_products['predicted_google_title'] = predicted_google_titles
matched_products['best_match_score'] = best_match_scores

dot_products = np.sum(amazon_matrix * google_matrix, axis=1)

# 3. Calculate the magnitudes (norms) of each vector
norm_amazon = np.linalg.norm(amazon_matrix, axis=1)
norm_google = np.linalg.norm(google_matrix, axis=1)

# 4. Calculate cosine similarity (safely handling division by zero)
denominators = norm_amazon * norm_google
pair_scores = np.divide(
    dot_products, 
    denominators, 
    out=np.zeros_like(dot_products), 
    where=denominators != 0
)

# 5. Assign the results to the new column
matched_products['pair_score'] = pair_scores

# 6. Validate the match (Does the predicted ID match the actual ground truth ID?)
matched_products['is_correct_match'] = matched_products['idGoogleBase'] == matched_products['predicted_idGoogleBase']

# Calculate and display performance metrics
execution_time = end_time - start_time
total_comparisons = amazon_matrix.shape[0] * google_matrix.shape[0]
accuracy = matched_products['is_correct_match'].mean() * 100



print("-" * 40)
print(f"Execution time: {execution_time:.4f} seconds")
print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
print(f"Model Accuracy: {accuracy:.2f}% ({matched_products['is_correct_match'].sum()} correct out of {len(matched_products)})")
print("-" * 40)

# View the results to see where the model succeeded and failed
columns_to_show = [
    'idAmazon', 
    'idGoogleBase', 
    'predicted_idGoogleBase', 
    'is_correct_match', 
    'best_match_score',
    'pair_score'
]

matched_products[columns_to_show].head(10)

In [ ]:
# Filter for records where the match was incorrect (False)
# We use .copy() so we can modify this new dataframe later without pandas warnings
not_matched_dataset = matched_products[matched_products['is_correct_match'] == False].copy()

# Alternatively, you can use the ~ (NOT) operator which is slightly more Pythonic:
# not_matched_dataset = matched_products[~matched_products['is_correct_match']].copy()

print(f"Total records: {len(matched_products)}")
print(f"Failed matches: {len(not_matched_dataset)}")

# View the failed matches to investigate why the model got them wrong
columns_to_show = [
    'idAmazon', 
    'predicted_idGoogleBase', 
    'idGoogleBase', 
    'best_match_score'
]
not_matched_dataset[columns_to_show].head()

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

# 1. Stack the embedding lists into 2D NumPy arrays (Matrices)
# Amazon matrix shape: (N_amazon, 768), Google matrix shape: (N_google, 768)
amazon_matrix = np.vstack(matched_products['amazon_embedding'].values)
google_matrix = np.vstack(matched_products['google_embedding'].values)

print("Calculating similarities... this will only take a second!")

# Start the high-resolution timer
start_time = time.perf_counter()
# 2. Calculate pairwise cosine similarity for all combinations at once
# This creates an N x M matrix where rows are Amazon products, columns are Google products
similarity_matrix = cosine_similarity(amazon_matrix, google_matrix)

# Perform the dot product using matrix multiplication
# .T transposes the google matrix to shape (768, N_google)
#similarity_matrix = amazon_matrix @ google_matrix.T

# Calculate pairwise Euclidean distances
#similarity_matrix = euclidean_distances(amazon_matrix, google_matrix)

# Stop the timer
end_time = time.perf_counter()

# Calculate and display results
execution_time = end_time - start_time
total_comparisons = amazon_matrix.shape[0] * google_matrix.shape[0]

print(f"Execution time: {execution_time:.4f} seconds")
print(f"Total pairwise comparisons calculated: {total_comparisons:,}")
print(f"Resulting matrix shape: {similarity_matrix.shape}")

# 3. Find the index of the highest similarity score for each Amazon row
best_match_indices = np.argmax(similarity_matrix, axis=1)

# 4. Extract the actual highest similarity score for each row
best_match_scores = np.max(similarity_matrix, axis=1)

# 5. Get the corresponding Google records using the indices
# We reset the index so it aligns perfectly with the Amazon DataFrame
best_google_matches = matched_products.iloc[best_match_indices].reset_index(drop=True)

# 6. Assign the best match details back to the original Amazon DataFrame
matched_products['idGoogleBaseMatch'] = best_google_matches['idGoogleBase'].values
matched_products['best_match_google_title'] = best_google_matches['google_title'].values
matched_products['best_match_score'] = best_match_scores

# View the results
# Keeping only relevant columns for a cleaner display
columns_to_show = ['idAmazon', 'amazon_title', 'idGoogleBase', 'best_match_google_title', 'best_match_score']
amazon[columns_to_show].head(10)



The last table contains a matching of product information on both website corresponding to the same product, but possibly described differently on the two websites:

From this raw data, we have pre-generated for you an eval and test set:

```python
../data/product_matching_eval.csv
../data/product_matching_test.csv
```


We will use the eval set to design the prompt and use the test set to evaluate the Gemini API performance on this prompt. This way, the performance we report will be closer to the real performance on never-seen-pairs of product descriptions.  


To genrate the eval and test split, we used the function in the cell below. It takes a sample (whose size is controlled by `SAMPLE_SIZE`) of matching product ID's in the `matching` dataframe, and it joins the information of the Google and Amazon product information contained in the `google` and `amazon` dataframes. Then it extracts pairs of matching Google and Amazon descriptions and creates a table of matching descriptions with columns `description_1` (Google), `description_2` (Amazon), and a target column named `match` whose value is set to `yes` since we have only matching pairs so far.
To create description pairs of not matching product, we permutate the second description columns while keeping the first description fixed, and concatenate this new dataframe of not matching descriptions to the one of matching description. We shuffle and then split the resulting table into two equal sized dataframes, which we save on disk as our eval and test splits. 

Observe that we only use the `title` column as product description. So there is much more info in the raw data. Nevertheless, we will see that Gemini will achieve a performance close to 100% on the test set. Remarkable!

**Note:** Uncomment the last line if you want to re-generate the eval and test set on a different sample of the data.

In [ ]:
import pandas as pd

# 1. Load the ground truth perfect mapping dataset
matching = pd.read_csv("../data/Amzon_GoogleProducts_perfectMapping.csv.gz")

# 2. Merge your predictions (the 'amazon' dataframe) with the ground truth
# how='left' keeps all your predictions.
# indicator=True adds a special '_merge' column to tell us where the row came from.
validation_df = amazon.merge(
    matching[['idAmazon', 'idGoogleBase']], 
    on=['idAmazon', 'idGoogleBase'], 
    how='left', 
    indicator=True
)

# 3. Split the data into the two requested datasets based on the indicator
# 'both' means the id combination existed in both dataframes (Correct Match)
matches_dataset = validation_df[validation_df['_merge'] == 'both'].copy()

# 'left_only' means the id combination was only in your predictions (Incorrect Match)
not_matched_dataset = validation_df[validation_df['_merge'] == 'left_only'].copy()

# 4. Clean up the datasets by dropping the temporary '_merge' column
matches_dataset.drop(columns=['_merge'], inplace=True)
not_matched_dataset.drop(columns=['_merge'], inplace=True)

# 5. Calculate counts and overall accuracy
match_count = len(matches_dataset)
not_match_count = len(not_matched_dataset)
total_predictions = len(amazon)
accuracy = (match_count / total_predictions) * 100

# Display the validation results
print(f"Total predictions made: {total_predictions}")
print(f"Correct matches (matches_dataset): {match_count}")
print(f"Incorrect matches (not_matched_dataset): {not_match_count}")
print(f"Overall Accuracy: {accuracy:.2f}%")

In [ ]:
SAMPLE_SIZE = 1000


def generate_test_and_eval_sets(sample_size=SAMPLE_SIZE):
    sample = matching.sample(sample_size, random_state=42)

    # Join the product information to the df of matching ID's
    matched_products = sample.merge(
        right=amazon, how="left", on="idAmazon"
    ).merge(right=google, how="left", on="idGoogleBase")

    google_descriptions = list(matched_products["google_title"])
    amazon_descriptions = list(matched_products["amazon_title"])

    # Create the dataframe of matching descriptions
    matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions,
            "match": "yes",
        }
    )

    # Create the dataframe of not matching descriptions
    amazon_descriptions_perm = [
        amazon_descriptions[i - 1] for i in range(len(amazon_descriptions))
    ]
    not_matching_descriptions = pd.DataFrame(
        {
            "description_1": google_descriptions,
            "description_2": amazon_descriptions_perm,
            "match": "no",
        }
    )

    full_dataset = pd.concat(
        [matching_descriptions, not_matching_descriptions], axis=0
    )

    # full_dataset = pd.concat(
    #     [matching_descriptions, not_matching_descriptions], axis=0
    # ).sample(len(matching_descriptions) * 2)

    # evalset = full_dataset[:sample_size]
    # testset = full_dataset[sample_size : 2 * sample_size]
    # evalset.to_csv("../data/product_matching_eval.csv", index=None)
    # testset.to_csv("../data/product_matching_test.csv", index=None)
    full_dataset.to_csv("../data/full_dataset.csv", index=None)


# Uncomment to generate a different data sample
generate_test_and_eval_sets()

The next cell loads the eval and test datasets that we pre-generated. The two CSV files contain 60 examples of product description pairs. Each pair is labeled with a `match` value of `yes` if the descriptions describe the same product and `no` otherwise. `description_1` comes from product title on Google while `description_2` comes from Amazon products.

In [ ]:
#evalset = pd.read_csv("../data/product_matching_eval.csv")
#testset = pd.read_csv("../data/product_matching_test.csv")
full_dataset = pd.read_csv("../data/full_dataset.csv")

Let's have a quick look at a few entries in this dataset:

In [ ]:
full_dataset.head()

Both splits are roughly balanced. To make sure that both splits are roughly balanced, we count the number of class instances for each split.

In [ ]:
full_dataset.match.value_counts()

Finding Mismatches Error Analysis

In [ ]:
# 1. Sort the dataset by cosine similarity in descending order (highest similarity first)
df_sorted = df.sort_values(by='cos_sim_768', ascending=False)

# 2. Slice the first 1000 records
top_similar = df_sorted.head(SAMPLE_SIZE)

# 3. Find mismatches (False Positives)
# Since these are the highest similarity scores, we expect them to be matches ('yes'). 
# Any record here with a 'no' is a mismatch between the model's prediction and ground truth.
mismatches = top_similar[top_similar['match'] == 'no']

# 4. Display the results
print(f"--- Error Analysis ---")
print(f"Found {len(mismatches)} mismatches (False Positives) in the top {SAMPLE_SIZE} most similar records.\n")

# Select the most relevant columns to display so you can read the text and see why it failed
columns_to_show = ['description_1', 'description_2', 'cos_sim_768', 'match']

mismatches[columns_to_show]

In [ ]:
# 5. Get the 1000 records with the LOWEST cosine similarity
# Since df_sorted is sorted descending (highest first), we can just take the tail
bottom_similar = df_sorted.tail(SAMPLE_SIZE)

# Alternatively, you could sort ascending:
# bottom_1000 = df.sort_values(by='cosine_similarity', ascending=True).head(1000)

# 6. Find mismatches (False Negatives)
# We expect these lowest-scoring records to be 'no'. 
# If the ground truth is 'yes', the model failed to recognize a true match.
false_negatives = bottom_similar[bottom_similar['match'] == 'yes']

# 7. Display the results
print(f"\n--- Error Analysis: False Negatives ---")
print(f"Found {len(false_negatives)} mismatches (False Negatives) in the 1000 least similar records.\n")
false_negatives[columns_to_show]

In [ ]:
# ---------------------------------------------------------
# NEW CODE: Calculate Mismatches and Generate Table
# ---------------------------------------------------------

print("\nCalculating mismatches across all configurations...")

# 1. Count the exact number of true matches ('yes') in the dataset
total_actual_yes = (df['match'] == 'yes').sum()

# 2. Define the metrics and their sorting logic
# similarity = higher is better (descending), distance = lower is better (ascending)
metrics = [
    {'name': 'Cosine Similarity', 'col_prefix': 'cos_sim', 'higher_is_better': True},
    {'name': 'Dot Product', 'col_prefix': 'dot', 'higher_is_better': True},
    {'name': 'Euclidean Distance', 'col_prefix': 'euc', 'higher_is_better': False}
]

vector_params = ['768', '128_unnorm', '128_norm']

param_mapping = {
    '768': '768 (Default)',
    '128_unnorm': '128 (Not Normalized)',
    '128_norm': '128 (L2 Normalized)'
}

results = []

# 3. Calculate mismatches for every combination, building row by row
for param in vector_params:
    # Initialize the row dictionary with the vector property name
    row_data = {'Vector Property': param_mapping[param]}
    
    for metric in metrics:
        col_name = f"{metric['col_prefix']}_{param}"
        
        # Sort: Descending for similarity, Ascending for distance
        df_sorted = df.sort_values(by=col_name, ascending=not metric['higher_is_better'])
        
        # Split into predicted Yes (top K) and predicted No (the rest)
        top_predictions = df_sorted.head(total_actual_yes)
        bottom_predictions = df_sorted.iloc[total_actual_yes:]
        
        # Count errors
        false_positives = (top_predictions['match'] == 'no').sum()
        false_negatives = (bottom_predictions['match'] == 'yes').sum()
        total_mismatches = false_positives + false_negatives
        
        # Assign the result to the corresponding metric column for this row
        row_data[metric['name']] = total_mismatches
        
    results.append(row_data)

# 4. Create the DataFrame directly from the list of row dictionaries
results_df = pd.DataFrame(results)

# Set the vector property column as the index for a cleaner table display
results_df.set_index('Vector Property', inplace=True)

# Display the final table
print("\n=== TOTAL MISMATCHES (False Positives + False Negatives) ===")
print(f"Evaluated by checking if the top {total_actual_yes} highest-scoring pairs are true matches.\n")
print(results_df.to_string())

In [ ]:
import numpy as np
import timeit

# Define the dataset size and the dimensions we want to test
n_vectors = 1000
dimensions_to_test = [128, 768]
iterations = 100  # Number of times to run each test for a stable average

print(f"Benchmarking pairwise operations for 2 arrays of {n_vectors} vectors.")
print(f"Results averaged over {iterations} iterations.")
print("=" * 70)
print(f"{'Dimensions':<12} | {'Dot Product':<15} | {'Euclidean':<15} | {'Cosine Sim':<15}")
print("-" * 70)

# Define the algorithms using highly optimized NumPy operations
# We use strings so they can be passed directly into the timeit module for isolated execution

# 1. Dot Product: Standard matrix multiplication
dot_product_code = "A @ B.T"

# 2. Euclidean Distance: Derived algebraically via (A-B)^2 = A^2 + B^2 - 2AB
euclidean_code = """
A_sq = np.sum(A**2, axis=1, keepdims=True)
B_sq = np.sum(B**2, axis=1)
# We use np.maximum to avoid negative zeros caused by floating point precision
np.sqrt(np.maximum(A_sq + B_sq - 2 * (A @ B.T), 0.0))
"""

# 3. Cosine Similarity: Dot product divided by the product of their norms
cosine_code = """
norm_A = np.linalg.norm(A, axis=1, keepdims=True)
norm_B = np.linalg.norm(B, axis=1)
(A @ B.T) / (norm_A * norm_B)
"""

# Loop through the target dimensions and run the benchmarks
for dim in dimensions_to_test:
    # Setup the environment and data for the specific dimension
    # Using a fixed seed ensures the random arrays are identical if run multiple times
    setup_code = f"""
import numpy as np
rng = np.random.default_rng(42)
A = rng.random(({n_vectors}, {dim}))
B = rng.random(({n_vectors}, {dim}))
"""

    # timeit.timeit returns total time for all iterations, divide by iterations for average
    dot_time = timeit.timeit(stmt=dot_product_code, setup=setup_code, number=iterations) / iterations
    euclidean_time = timeit.timeit(stmt=euclidean_code, setup=setup_code, number=iterations) / iterations
    cosine_time = timeit.timeit(stmt=cosine_code, setup=setup_code, number=iterations) / iterations
    
    # Convert times from seconds to milliseconds for easier reading
    dot_ms = dot_time * 1000
    euc_ms = euclidean_time * 1000
    cos_ms = cosine_time * 1000

    # Print the row for this dimension
    print(f"{dim:<12} | {dot_ms:>10.2f} ms | {euc_ms:>10.2f} ms | {cos_ms:>10.2f} ms")

print("=" * 70)

In [ ]:
import numpy as np
import timeit

# Define the dataset size
n_vectors = 1000
dimensions = 128  # Typical size for basic embeddings
iterations = 100  # Number of times to run each test for a stable average

# 1. Setup the environment and data
setup_code = f"""
import numpy as np
rng = np.random.default_rng(42)
A = rng.random(({n_vectors}, {dimensions}))
B = rng.random(({n_vectors}, {dimensions}))
"""

# 2. Define the algorithms using highly optimized NumPy operations

# Dot Product: Just matrix multiplication
dot_product_code = "A @ B.T"

# Euclidean Distance: Derived algebraically via (A-B)^2 = A^2 + B^2 - 2AB
euclidean_code = """
A_sq = np.sum(A**2, axis=1, keepdims=True)
B_sq = np.sum(B**2, axis=1)
# We use np.maximum to avoid negative zeros caused by floating point precision
np.sqrt(np.maximum(A_sq + B_sq - 2 * (A @ B.T), 0.0))
"""

# Cosine Similarity: Dot product divided by the product of their norms
cosine_code = """
norm_A = np.linalg.norm(A, axis=1, keepdims=True)
norm_B = np.linalg.norm(B, axis=1)
(A @ B.T) / (norm_A * norm_B)
"""

# 3. Run the benchmarks
print(f"Benchmarking pairwise operations for 2 arrays of {n_vectors} vectors ({dimensions}D)...")
print("-" * 50)

# timeit.timeit returns total time for all iterations, so we divide by iterations for average time per run
dot_time = timeit.timeit(stmt=dot_product_code, setup=setup_code, number=iterations) / iterations
euclidean_time = timeit.timeit(stmt=euclidean_code, setup=setup_code, number=iterations) / iterations
cosine_time = timeit.timeit(stmt=cosine_code, setup=setup_code, number=iterations) / iterations

print(f"Dot Product:        {dot_time * 1000:.2f} milliseconds")
print(f"Euclidean Distance: {euclidean_time * 1000:.2f} milliseconds")
print(f"Cosine Similarity:  {cosine_time * 1000:.2f} milliseconds")

In [ ]:
# ---------------------------------------------------------
# NEW CODE: Calculate Mismatches and Generate Table
# ---------------------------------------------------------

print("\nCalculating mismatches across all configurations...")

# 1. Count the exact number of true matches ('yes') in the dataset
total_actual_yes = (df['match'] == 'yes').sum()

# 2. Define the metrics and their sorting logic
# similarity = higher is better (descending), distance = lower is better (ascending)
metrics = [
    {'name': 'Cosine Similarity', 'col_prefix': 'cos_sim', 'higher_is_better': True},
    {'name': 'Dot Product', 'col_prefix': 'dot', 'higher_is_better': True},
    {'name': 'Euclidean Distance', 'col_prefix': 'euc', 'higher_is_better': False}
]

vector_params = ['768', '128_unnorm', '128_norm']
results = []

# 3. Calculate mismatches for every combination
for param in vector_params:
    for metric in metrics:
        col_name = f"{metric['col_prefix']}_{param}"
        
        # Sort: Descending for similarity, Ascending for distance
        df_sorted = df.sort_values(by=col_name, ascending=not metric['higher_is_better'])
        
        # Split into predicted Yes (top K) and predicted No (the rest)
        top_predictions = df_sorted.head(total_actual_yes)
        bottom_predictions = df_sorted.iloc[total_actual_yes:]
        
        # Count errors
        false_positives = (top_predictions['match'] == 'no').sum()
        false_negatives = (bottom_predictions['match'] == 'yes').sum()
        total_mismatches = false_positives + false_negatives
        
        results.append({
            'Vector Size': param,
            'Metric': metric['name'],
            'Mismatches': total_mismatches
        })

# 4. Format into a Pandas Pivot Table
results_df = pd.DataFrame(results)

# Make the column names user-friendly
param_mapping = {
    '768': '768 (Default)',
    '128_unnorm': '128 (Not Normalized)',
    '128_norm': '128 (L2 Normalized)'
}
results_df['Vector Size'] = results_df['Vector Size'].map(param_mapping)

# Pivot the data to create a 3x3 grid
pivot_table = results_df.pivot(index='Metric', columns='Vector Size', values='Mismatches')

# Enforce a logical column order for readability
pivot_table = pivot_table[['768 (Default)', '128 (Not Normalized)', '128 (L2 Normalized)']]

# Display the final table
print("\n=== TOTAL MISMATCHES (False Positives + False Negatives) ===")
print(f"Evaluated by checking if the top {total_actual_yes} highest-scoring pairs are true matches.\n")
pivot_table

#  Model implementation

We start by instanciating our client using the Gen AI SDK. We'll use the `gemini-2.5-flash` version of Gemini which is a large language model (LLM) developed by Google.

In [ ]:
MODEL = "gemini-2.5-flash"

client = genai.Client(vertexai=True, location="us-central1")

Using this client, we implement in the next cell a simple function that takes two product descriptions and a parameterized prompt as input, and outputs `yes` or `no` depending on whether the Gemini model thinks the product descriptions are matching.

In [ ]:
def are_products_matching(d1, d2, prompt):
    prompt = prompt.format(desc1=d1, desc2=d2)
    generation_config = GenerateContentConfig(
        temperature=0.1,
        top_p=0.8,
        top_k=40,
    )
    answer = client.models.generate_content(
        model=MODEL, contents=prompt, config=generation_config
    )
    return answer.text.strip()

The next cell allows us to rapidly test on the evaluation set whether a given prompt seems to be working for this use case.

In [ ]:
PROMPT = """
Are the following two descriptions describing the same product?

EXAMPLE:

Description 1: {desc1}
Description 2: {desc2}

If they are similar, reply with "yes" and if they are not similar, reply with "no".
Answer only with "yes" or "no"
"""

index = random.randint(0, len(false_negatives) - 1)

d1 = evalset.iloc[index]["description_1"]
d2 = evalset.iloc[index]["description_2"]
ground_truth = evalset.iloc[index]["match"]
prediction = are_products_matching(d1, d2, prompt=PROMPT)


print(
    f"""
Are the following two descriptions describing the same product?

Description 1: {d1}

Descriptions 2: {d2}

MODEL ANSWER: {prediction}
GROUND TRUTH: {ground_truth}
"""
)

# Model Analysis

We now analyze the performance of our model on the test set.

Large language models may sometimes output something other than "yes" or "no," even if we ask them politely to do so. This could be due to safety filters being triggered or the model simply not understanding the question. Therefore, we first need to determine the proportion of requests that our model fails to answer. In this case, it is around 6%, which may be acceptable. Further prompt engineering or model tuning could help to reduce this number.

The second aspect to consider is the performance of the model on the requests
that succeded (i.e. for whose the output was actually `yes` or `no`).
Since the test set is balanced, we can compute the model accuracy, which is 98%. This means that only a single example in the test set received a prediction that was different from the ground truth.

## Scoring the test set

To simplify evaluation, we implement a function in the next cell that will add a `prediction` column to our `testset`. This column will contain the predictions received from the Gemini API:

In [ ]:
def apply_prompt(prompt, dataset):
    scored_dataset = dataset.copy()
    scored_dataset["predictions"] = scored_dataset.apply(
        lambda row: are_products_matching(
            row.description_1, row.description_2, prompt=PROMPT
        ),
        axis=1,
    )
    return scored_dataset

Let's apply this function to our `testset` using our simple prompt that we designed on the `evalset`:

In [ ]:
scored_testset = apply_prompt(prompt=PROMPT, dataset=false_negatives)

Since requests to the Gemini API are limited and capped to 60 requests per minute, we save our scoring to disk so that we can analyze it offline if needed.

In [ ]:
scored_testset.to_csv("scored_evalset.csv", index=False)

Here are the predictions of our model. We can see that most examples are classified correctly, although some requests failed, resulting in empty predictions. We will need to analyze these cases separately and compute the accuracy only for the requests that succeeded:

In [ ]:
scored_testset.head(10)

### Failed predictions

The cell below list the number of failed predictions, that is, predictions which are anything other that `yes` or `no`. There are several possible causes to such a behavior. Gemini, as all LLM's, has been trained to predict the next most likely word from a sequence of words. Therefore, although we instruct Gemini explicitely in our prompt to answer by `yes` or `no`, it may happen that certain product descriptions confuse Gemini, resulting in something different from `yes` or `no`. Another issue is that the language in the product descriptions may trigger a safety filter, which then will replace Gemini raw answer by some standard warning text. These safety filters can be triggered by words in the product descriptions that are too evocating of health or medical issues for instance, or violence, among many [other safety settings](https://developers.generativeai.google/guide/safety_setting). 

In [ ]:
failed_predictions_mask = ~(
    scored_testset.predictions.str.match("yes")
    | scored_testset.predictions.str.match("no")
)

failed_predictions_mask.value_counts()

In [ ]:
proportion_of_failed_predictions = failed_predictions_mask.sum() / len(
    failed_predictions_mask
)
print(
    "Proportion of failed requests:",
    round(proportion_of_failed_predictions, 3) * 100,
    "%",
)

The next cell examines the failed requests. Some of the terms may have triggered safety filters, but this would require further investigation:

In [ ]:
scored_testset[failed_predictions_mask]

## Accuracy on succeeded predictions

Let us now compute the model accuracy on the requests that succeeded. First, we remove all the failed requests from the test set:

In [ ]:
scored_testset_with_predictions = scored_testset[~failed_predictions_mask]

Then, we compute the number of correct answers:

In [ ]:
is_prediction_correct = (
    scored_testset_with_predictions.match
    == scored_testset_with_predictions.predictions
)
is_prediction_correct.value_counts()

The accuracy of our model is the number of correct answers divided by the number of predictions that succeeded, which we compute below:


In [ ]:
model_accuracy = is_prediction_correct.sum() / len(is_prediction_correct)
print("Model accuracy:", round(model_accuracy, 3))

Actually, for our small test set, this corresponds to our model having made the following number of errors:

In [ ]:
uncorrect_predictions_mask = (
    scored_testset_with_predictions.match
    != scored_testset_with_predictions.predictions
)
sum(uncorrect_predictions_mask)

We can now inspect the errors:

In [ ]:
scored_testset_with_predictions[uncorrect_predictions_mask]

Copyright 2023 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License